# 逻辑回归（Logistic Regression）通用模板 · Python 3.11 · 美赛直接复用

**适用**：二分类 / 多分类判别、影响因素分析、事件发生概率预测。

## 模型与论文可引用的公式

- 对数几率线性模型：$\ln\dfrac{p}{1-p}=\beta_0+\beta_1x_1+\cdots+\beta_mx_m$，
  反解得 $p=\dfrac{1}{1+e^{-(\beta_0+\beta^\top x)}}$（sigmoid）。
- 参数用极大似然估计：$L=\prod_i p_i^{y_i}(1-p_i)^{1-y_i}$，
  等价最小化对数损失 $-\sum_i[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$。
- **优势比 OR** $=\exp(\beta_j)$：其他变量不变时 $x_j$ 每增加 1 个标准化单位，
  事件发生比（odds）变为原来的 $\exp(\beta_j)$ 倍；$>1$ 促进、$<1$ 抑制。
- **Wald 检验**：$\hat\beta$ 的协方差 $\approx (X^\top W X)^{-1}$，
  $W=\mathrm{diag}[p_i(1-p_i)]$；$z=\hat\beta/SE$，
  $p=2\Phi(-|z|)$，OR 的 95% 置信区间 $=\exp(\hat\beta\pm1.96SE)$。
- **VIF** $=\dfrac{1}{1-R_j^2}$：用其余特征回归 $x_j$ 得到；$>10$ 提示严重共线，
  共线会让系数和 OR 不稳定（此时优先看预测指标而非单个系数）。
- 评估：分层 $k$ 折交叉验证 + 留出集，Accuracy / Macro-F1 / ROC-AUC、
  混淆矩阵、ROC 曲线；二分类时再按 **F1 最大** 与 **Youden J 最大** 各给一个最优阈值。

> **避免虚拟变量陷阱**：一组独热编码列之和恒为 1，全部保留会造成完全共线，
> 使 VIF 变成无穷大、Wald 标准误失真。因此本模板默认 `onehot_drop_first=True`，
> 每组丢掉第一类作为**参照组**，系数解释为「相对参照组的变化」。
>
> 依赖：`numpy pandas scipy scikit-learn matplotlib openpyxl`（Python 3.11 已实测）。
> 本 Notebook 自包含，不依赖同目录其他 Notebook。

In [2]:
# ============================================================

# 0. 环境准备（Python 3.11）

# 本 Notebook 自包含：不 import 其他 Notebook，换题目直接整份复制走

# ============================================================

import sys

import warnings

import platform

from pathlib import Path

from dataclasses import dataclass, field, fields, replace



import numpy as np

import pandas as pd

import matplotlib

import matplotlib.pyplot as plt

from scipy import stats

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster



from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.decomposition import PCA as SkPCA

from sklearn.cluster import KMeans

from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.metrics import (

    silhouette_score, silhouette_samples,

    calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score,

    accuracy_score, precision_score, recall_score, f1_score,

    roc_auc_score, roc_curve, precision_recall_curve,

    confusion_matrix, classification_report, ConfusionMatrixDisplay,

)

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from sklearn.base import clone

from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype



if sys.version_info[:2] < (3, 11):

    print(f"[警告] 当前 Python {platform.python_version()}，本模板按 3.11 编写。")

warnings.filterwarnings("ignore", category=FutureWarning)

warnings.filterwarnings("ignore", category=UserWarning)



plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun", "Arial Unicode MS"]

plt.rcParams["axes.unicode_minus"] = False

plt.rcParams["figure.dpi"] = 110

plt.rcParams["axes.grid"] = True

plt.rcParams["grid.alpha"] = 0.3

pd.set_option("display.unicode.east_asian_width", True)

pd.set_option("display.max_columns", 40)

pd.set_option("display.width", 160)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")



import sklearn

print("Python", platform.python_version(), "| numpy", np.__version__,

      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

Python 3.11.9 | numpy 2.4.6 | pandas 3.1.0 | sklearn 1.9.1


### 换一份数据，怎么复用？（照抄即可）



1. 把新文件放进 `导入数据/`（`.csv` / `.xlsx` / `.xls` 都支持）；

2. 在 Notebook 末尾新建一格，只写一行：



```python

bundle = run_on("你的文件名.csv", target="标签列名", tag="本次名字")

```



3. `Run All`。结果自动写进 `输出数据/`，文件名形如

`逻辑回归_示例_结果汇总.xlsx`（前缀固定，中间是 tag）：CSV 明细 + 多 Sheet Excel + 图片。



**数据格式**：一行一个样本、一列一个指标，中文列名没问题。BOM、前后空格、

文本数字、`999` 哨兵、重复行、缺失值、标准化、文字类别转独热编码全部自动处理。



**只有 3 个值要填**：文件名、`target`（`必填：逻辑回归是有监督模型，需要一列标签；连续标签会自动二分或分箱`）、`tag`（换数据就改名，否则覆盖上一份）。

其余参数（`id_cols` / `features` / `na_markers` / `scale` / `k_max` 等）只在需要时改。

`run_on` 也接受任意 Config 字段作关键字参数，例如 `make_plots=False, k_max=10`。



> 输出文件名固定、同名覆盖：再跑 10 次也不会多出文件；换了 `tag` 才会新增一组。

> 想跳过末尾示例就不动 `RUN_EXAMPLES=False`；想一张图都不生成用 `make_plots=False`；

> 清旧结果用 `clean_outputs("旧tag")`。**重新打开 Notebook 才会读到磁盘上的最新版。**

In [4]:
# ============================================================

# 1. 路径 + 配置区（换数据只改这里，或直接用后面的 run_on 一行搞定）

# ============================================================

PREFIX = "逻辑回归"          # 本 Notebook 所有输出文件的固定前缀



def find_project_root(marker="导入数据"):

    """从当前目录向上找含 导入数据/ 的项目根，兼容从子目录或根目录启动内核。"""

    for base in [Path.cwd(), *Path.cwd().parents]:

        if (base / marker).is_dir():

            return base

    return Path.cwd()



ROOT = find_project_root()

DATA_DIR = ROOT / "导入数据" if (ROOT / "导入数据").is_dir() else ROOT

OUT_DIR = ROOT / "输出数据" if (ROOT / "输出数据").is_dir() else ROOT

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录 :", ROOT)

print("输出目录   :", OUT_DIR)





@dataclass

class Config:

    """所有建模开关集中在这里；标 ★ 的是换数据时通常要改的。"""

    data_path: object = None        # ★ 数据文件；None = 自动找 导入数据/ 里第一个表格

    sheet_name: object = 0          # Excel 工作表（名字或序号）

    id_cols: tuple = ("date",)      # ★ 编号/日期等不进模型的列

    target: object = None           # ★ 有监督标签列；None = 无监督（KMeans/PCA 不填）

    features: object = None         # 参与建模的列；None = 自动推断

    tag: str = "demo"               # ★ 输出文件名前缀，换数据记得改名

    make_plots: bool = True         # False = 一张图都不生成（连画布都不创建）



    # ---- 读入与清洗 ----

    na_markers: tuple = (999, 999.0, -999, "未填", "NA", "N/A", "null", "未知")

    drop_duplicates: bool = True

    drop_high_missing: bool = True

    missing_threshold: float = 0.50  # 缺失率超过它就不进模型



    # ---- 预处理 ----

    impute_num: str = "median"      # median / mean / zero

    impute_cat: str = "mode"        # mode / unknown

    add_missing_indicator: bool = True

    outlier: str = "winsor_iqr"     # winsor_iqr / winsor_3sigma / none

    scale: str = "standard"         # standard / minmax / robust / none

    max_onehot_cardinality: int = 30

    onehot_drop_first: bool = False

    random_state: int = 42          # 全局随机种子，保证结果可复现


    # ---- 逻辑回归专有 ----
    max_label_classes: int = 20    # 数值标签取值超过它 → 自动分箱
    lr_binary_split: bool = True   # True: 连续标签按中位数二分成 低/高（便于算 OR）
    lr_C: float = 1.0              # L2 正则强度倒数，越小正则越强
    lr_class_weight: object = "balanced"  # 类别不平衡时自动加权；可设 None
    n_splits: int = 5
    test_size: float = 0.25




CONFIG = Config(data_path=DATA_DIR / "mcm_final_raw.csv", target="survey_score", tag="满意度高低", onehot_drop_first=True)

RUN_EXAMPLES = False                # True 才会执行末尾的可选示例（多跑一组数据、多产一组文件）





def save_fig(fig, name, cfg):

    """统一保存图片：文件名 = {PREFIX}_{tag}_{图名}.png，同名覆盖；用完关闭画布。"""

    if not getattr(cfg, "make_plots", True):

        plt.close(fig)

        return None

    path = OUT_DIR / f"{PREFIX}_{cfg.tag}_{name}.png"

    fig.savefig(path, dpi=150, bbox_inches="tight")

    plt.show()

    plt.close(fig)                  # 释放 figure，反复 Run All 不会堆积内存

    print(f"[图片] {path.name}（同名覆盖）")

    return path





def clean_outputs(tag=None):

    """清理某个 tag 遗留的产物（只匹配本模板前缀，不碰其他文件）。"""

    tag = CONFIG.tag if tag is None else tag

    files = sorted(OUT_DIR.glob(f"{PREFIX}_{tag}_*"))

    for f in files:

        f.unlink()

    print(f"已清理 tag='{tag}' 的 {len(files)} 个文件")

    return files





print("配置：", CONFIG)

print("运行末尾示例：", RUN_EXAMPLES)

项目根目录 : D:\git项目
输出目录   : D:\git项目\输出数据
配置： Config(data_path=WindowsPath('D:/git项目/导入数据/mcm_final_raw.csv'), sheet_name=0, id_cols=('date',), target='survey_score', features=None, tag='满意度高低', make_plots=True, na_markers=(999, 999.0, -999, '未填', 'NA', 'N/A', 'null', '未知'), drop_duplicates=True, drop_high_missing=True, missing_threshold=0.5, impute_num='median', impute_cat='mode', add_missing_indicator=True, outlier='winsor_iqr', scale='standard', max_onehot_cardinality=30, onehot_drop_first=True, random_state=42, max_label_classes=20, lr_binary_split=True, lr_C=1.0, lr_class_weight='balanced', n_splits=5, test_size=0.25)
运行末尾示例： False


### 2. 通用数据读取 `load_table`



自动处理美赛附件的常见坑：CSV 编码/BOM、列名与字符串首尾空格、数字被存成文本、

`999` 等哨兵值、日期列、全空行列、重复行，并打印每一步改了什么（论文预处理段落可直接引用）。

In [6]:
TEXT_NA = ["", " ", "  ", "-", "?", "NA", "N/A", "na", "n/a", "null", "NULL",

           "None", "nan", "NaN", "未填", "未知", "缺失", "无"]





def resolve_data_path(cfg):

    """确定数据文件；相对路径按 导入数据/ 解析，没指定就自动挑一个表格。"""

    if cfg.data_path is not None:

        p = Path(cfg.data_path)

        if not p.is_absolute():

            p = (ROOT / p) if (p.parts and p.parts[0] == DATA_DIR.name) else (DATA_DIR / p)

        if not p.exists():

            raise FileNotFoundError(f"找不到数据文件：{p}\n请检查文件名或传入绝对路径。")

        return p

    cand = sorted([p for p in DATA_DIR.iterdir()

                   if p.suffix.lower() in (".csv", ".xlsx", ".xls")])

    if not cand:

        raise FileNotFoundError(f"{DATA_DIR} 里没有 csv/xlsx 文件。")

    print(f"[提示] 自动选用：{cand[0].name}")

    return cand[0]





def load_table(cfg):

    """读入 → 初清洗 → (DataFrame, 元信息)。数值哨兵留到 3.5 步替换以便统计个数。"""

    path = resolve_data_path(cfg)

    na_values = list(dict.fromkeys(TEXT_NA))

    if path.suffix.lower() in (".xlsx", ".xls"):

        df = pd.read_excel(path, sheet_name=cfg.sheet_name, na_values=na_values)

        enc = "Excel(openpyxl)"

    else:

        df, enc = None, None

        for e in ("utf-8-sig", "utf-8", "gbk", "gb18030"):

            try:

                df = pd.read_csv(path, encoding=e, na_values=na_values,

                                 skipinitialspace=True)

                enc = e

                break

            except UnicodeDecodeError:

                continue

        if df is None:

            raise UnicodeDecodeError("all", b"", 0, 1, "常见编码都无法解码该 CSV。")



    n_raw = len(df)

    # 列名：去 BOM / 去空格 / 重名加后缀

    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

    seen, cols = {}, []

    for c in df.columns:

        seen[c] = seen.get(c, 0) + 1

        cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")

    df.columns = cols

    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()



    # 字符串列：strip 后把文字型缺失标记置 NaN

    for c in df.columns:

        if not is_numeric_dtype(df[c]) and not is_datetime64_any_dtype(df[c]):

            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)

            df[c] = df[c].replace({m: np.nan for m in TEXT_NA})



    # 文本数字列转数值；名字带 date/time 且能解析的列转 datetime

    turned, date_cols = [], []

    for c in df.columns:

        if is_numeric_dtype(df[c]) or is_datetime64_any_dtype(df[c]):

            continue

        s = df[c]

        valid = s.notna().sum()

        if valid == 0:

            continue

        conv = pd.to_numeric(s, errors="coerce")

        if conv.notna().sum() >= 0.90 * valid:

            df[c] = conv.astype(float)

            turned.append(c)

        elif any(k in c.lower() for k in ("date", "time", "日期", "时间")):

            dt = pd.to_datetime(s, errors="coerce")

            if dt.notna().sum() >= 0.90 * valid:

                df[c] = dt

                date_cols.append(c)



    # 数值哨兵（999 等）转 NaN，逐列统计，便于论文说明处理依据

    hits = {}

    num_sentinels = [v for v in cfg.na_markers if isinstance(v, (int, float))]

    for c in df.select_dtypes(include="number").columns:

        hit = int(df[c].isin(num_sentinels).sum())

        if hit:

            hits[c] = hit

            df[c] = df[c].replace({v: np.nan for v in num_sentinels})



    n_dup = int(df.duplicated().sum())

    if cfg.drop_duplicates and n_dup:

        df = df.drop_duplicates().reset_index(drop=True)



    meta = dict(path=path, encoding=enc, n_raw=n_raw, n_final=len(df),

                n_duplicates=n_dup, numeric_turned=turned,

                date_cols=date_cols, sentinel_hits=hits)

    print(f"读取成功 : {path.name}（{enc}）")

    print(f"数据规模 : {n_raw} 行 → {len(df)} 行，{df.shape[1]} 列；删重复 {n_dup} 行")

    print(f"文本转数值 : {turned or '无'}；日期列：{date_cols or '无'}；哨兵转缺失：{hits or '无'}")

    return df, meta





df, META = load_table(CONFIG)

print("\n字段类型：")

print(df.dtypes.to_string())

读取成功 : mcm_final_raw.csv（utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删重复 15 行
文本转数值 : ['pm25']；日期列：['date']；哨兵转缺失：{'traffic_index': 6}

字段类型：
date                datetime64[us]
region                         str
pm25                       float64
temperature                float64
traffic_index              float64
fuel_type                      str
survey_score               float64
household_income           float64


### 3. 数据体检



建模前先看每列的缺失率、唯一值、极值与 IQR 异常点；缺失率过高的列会在预处理阶段自动剔除。

一张综合图（缺失率 + 标准化箱线图）便于写进论文的数据说明部分。

In [8]:
def data_audit(df, cfg, plot=True):

    """逐列体检表；plot=True 且 make_plots=True 时画一张综合图。"""

    rows = []

    for c in df.columns:

        s = df[c]

        miss = int(s.isna().sum())

        rec = dict(列名=c, 类型=str(s.dtype), 缺失数=miss,

                   缺失率=round(miss / len(df), 4), 唯一值=int(s.nunique(dropna=True)))

        if is_numeric_dtype(s):

            q1, q3 = s.quantile(0.25), s.quantile(0.75)

            iqr = q3 - q1

            rec.update(最小值=round(float(s.min()), 3) if s.notna().any() else np.nan,

                       中位数=round(float(s.median()), 3) if s.notna().any() else np.nan,

                       最大值=round(float(s.max()), 3) if s.notna().any() else np.nan,

                       IQR异常点=int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),

                       建议=("剔除(缺测过多)" if (cfg.drop_high_missing and

                                             miss / len(df) > cfg.missing_threshold)

                             else ("常数列,剔除" if s.nunique(dropna=True) <= 1 else "保留")))

        else:

            rec.update(最小值="", 中位数="", 最大值="", IQR异常点=0,

                       建议=("日期/标识列,不建模" if is_datetime64_any_dtype(s)

                             else "独热编码" if s.nunique(dropna=True) <= cfg.max_onehot_cardinality

                             else "高基数,剔除"))

        rows.append(rec)

    audit = pd.DataFrame(rows)

    print("===== 数据体检表 =====")

    print(audit.to_string(index=False))



    if plot and cfg.make_plots:

        num = [c for c in df.columns if is_numeric_dtype(df[c])]

        miss_tbl = audit[audit["缺失率"] > 0]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

        if len(miss_tbl):

            axes[0].barh(miss_tbl["列名"], miss_tbl["缺失率"] * 100, color="#d95f5f")

            axes[0].axvline(cfg.missing_threshold * 100, ls="--", color="#444")

        axes[0].set_title("各列缺失率 (%)"); axes[0].set_xlabel("%")

        if num:

            z = df[num].apply(lambda s: (s - s.mean()) / (s.std() or 1))

            axes[1].boxplot([z[c].dropna() for c in num])

            axes[1].set_xticks(range(1, len(num) + 1))

            axes[1].set_xticklabels(num, rotation=30, ha="right")

            axes[1].set_title("数值列分布（标准化后）")

        fig.tight_layout()

        save_fig(fig, "数据体检", cfg)

    return audit





audit = data_audit(df, CONFIG)

===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    survey_score        float64      18  0.0492     342    44.0990    69.7360    99.6440          3               保留
household_income        float64     318  0.8689      42 3,115.6080 5,240.4430 9,9

### 4. 可复用预处理器 `TabularPreprocessor`



`fit` 只在训练数据上统计中位数/众数、缩尾边界、标准化参数、类别清单，

`transform` 对新数据用**同一套参数**，避免数据泄漏。自动剔除标签列、日期列、

高缺失列、常数列、高基数文本列；数值列填补 → 缩尾 → 标准化，文字列众数填补 → 独热编码。

In [10]:
class TabularPreprocessor:

    """任意表格 → 数值特征矩阵 X（保留可读列名），支持 fit/transform 复用。"""



    def __init__(self, cfg):

        self.cfg = cfg

        self.numeric_cols_, self.categorical_cols_ = [], []

        self.indicator_cols_, self.dropped_cols_ = [], {}

        self.fill_num_, self.fill_cat_, self.bounds_ = {}, {}, {}

        self.center_, self.scale_, self.categories_ = {}, {}, {}

        self.feature_names_ = []



    def _pick_columns(self, df):

        excluded = set(self.cfg.id_cols or ())

        if self.cfg.target is not None:

            excluded.add(self.cfg.target)

        for c in df.columns:

            if is_datetime64_any_dtype(df[c]) or any(

                    k in str(c).lower() for k in ("date", "time", "日期", "时间")):

                excluded.add(c)

        cols = [c for c in df.columns if c not in excluded]

        if self.cfg.features is not None:

            cols = [c for c in self.cfg.features if c in df.columns]

        for c in cols:

            s = df[c]

            r = s.isna().mean()

            if r == 1:

                self.dropped_cols_[c] = "全空列"; continue

            if self.cfg.drop_high_missing and r > self.cfg.missing_threshold:

                self.dropped_cols_[c] = f"缺失率 {r:.1%}"; continue

            if s.nunique(dropna=True) <= 1:

                self.dropped_cols_[c] = "常数列"; continue

            if is_numeric_dtype(s):

                self.numeric_cols_.append(c)

            elif s.nunique(dropna=True) > self.cfg.max_onehot_cardinality:

                self.dropped_cols_[c] = "高基数文本列"

            else:

                self.categorical_cols_.append(c)

        if not self.numeric_cols_ and not self.categorical_cols_:

            raise ValueError("没有可用特征，请检查 features / missing_threshold。")



    def fit(self, df):

        self._pick_columns(df)

        cfg = self.cfg

        for c in self.numeric_cols_:

            s = pd.to_numeric(df[c], errors="coerce").astype(float)

            fill = (float(s.mean()) if cfg.impute_num == "mean" else

                    0.0 if cfg.impute_num == "zero" else float(s.median()))

            self.fill_num_[c] = fill if np.isfinite(fill) else 0.0

            if s.isna().any() and cfg.add_missing_indicator and f"{c}_missing" not in df.columns:

                self.indicator_cols_.append(c)

            f = s.fillna(self.fill_num_[c])

            if cfg.outlier == "winsor_iqr":

                q1, q3 = f.quantile(0.25), f.quantile(0.75)

                i = q3 - q1

                lo, hi = float(q1 - 1.5 * i), float(q3 + 1.5 * i)

            elif cfg.outlier == "winsor_3sigma":

                mu, sd = float(f.mean()), float(f.std())

                lo, hi = mu - 3 * sd, mu + 3 * sd

            else:

                lo, hi = -np.inf, np.inf

            self.bounds_[c] = (lo, hi)

            z = f.clip(lo, hi)

            if cfg.scale == "minmax":

                self.center_[c], self.scale_[c] = float(z.min()), float(z.max() - z.min())

            elif cfg.scale == "robust":

                self.center_[c] = float(z.median())

                self.scale_[c] = float(z.quantile(0.75) - z.quantile(0.25))

            elif cfg.scale == "standard":

                self.center_[c], self.scale_[c] = float(z.mean()), float(z.std())

            else:

                self.center_[c], self.scale_[c] = 0.0, 1.0

            if not np.isfinite(self.scale_[c]) or self.scale_[c] == 0:

                self.scale_[c] = 1.0

        for c in self.categorical_cols_:

            s = df[c]

            if cfg.impute_cat == "unknown":

                self.fill_cat_[c] = "未知"

            else:

                m = s.mode(dropna=True)

                self.fill_cat_[c] = str(m.iat[0]) if len(m) else "未知"

            cats = sorted({str(v).strip() for v in s.dropna().unique()})

            if cfg.onehot_drop_first and len(cats) > 1:

                cats = cats[1:]

            self.categories_[c] = cats

        names = list(self.numeric_cols_) + [f"{c}_missing" for c in self.indicator_cols_]

        for c in self.categorical_cols_:

            names += [f"{c}={v}" for v in self.categories_[c]]

        seen, out = {}, []

        for n in names:

            seen[n] = seen.get(n, 0) + 1

            out.append(n if seen[n] == 1 else f"{n}_{seen[n]}")

        self.feature_names_ = out

        return self



    def transform(self, df):

        miss = [c for c in self.numeric_cols_ + self.categorical_cols_ if c not in df.columns]

        if miss:

            raise KeyError(f"新数据缺少建模所需列：{miss}")

        blocks = []

        for c in self.numeric_cols_:

            s = pd.to_numeric(df[c], errors="coerce").astype(float).fillna(self.fill_num_[c])

            lo, hi = self.bounds_[c]

            blocks.append(((s.clip(lo, hi) - self.center_[c]) / self.scale_[c]).to_numpy(float))

        for c in self.indicator_cols_:

            blocks.append(df[c].isna().to_numpy(float))

        for c in self.categorical_cols_:

            s = df[c].map(lambda v: str(v).strip() if pd.notna(v) else self.fill_cat_[c])

            for v in self.categories_[c]:

                blocks.append((s == v).to_numpy(float))

        return pd.DataFrame(np.column_stack(blocks), columns=self.feature_names_, index=df.index)



    def fit_transform(self, df):

        return self.fit(df).transform(df)





pre = TabularPreprocessor(CONFIG).fit(df)

X = pre.transform(df)

print("数值列 :", pre.numeric_cols_)

print("类别列 :", pre.categorical_cols_)

print("剔除列 :", pre.dropped_cols_ or "无")

print("特征矩阵 X :", X.shape)

数值列 : ['pm25', 'temperature', 'traffic_index']
类别列 : ['region', 'fuel_type']
剔除列 : {'household_income': '缺失率 86.9%'}
特征矩阵 X : (366, 12)


### 5. 标签处理、VIF 共线性诊断与 Wald 系数表

系数 $\beta$、优势比 OR、95% 置信区间与显著性是逻辑回归在美赛里最常被要求解释的部分；
VIF 用来判断共线是否让系数失真。

In [12]:
# ============================================================
# 5. 标签处理 + VIF + Wald 系数表
# ============================================================
def prepare_target(df, cfg):
    """取标签 → 必要时分箱 → LabelEncoder 编码；缺失位置记为 -1 后续剔除。"""
    y_raw = df[cfg.target]
    if is_numeric_dtype(y_raw) and y_raw.nunique(dropna=True) > cfg.max_label_classes:
        if cfg.lr_binary_split:
            cut = float(y_raw.median())
            y_raw = y_raw.map(lambda v: np.nan if pd.isna(v) else ("高" if v >= cut else "低"))
            print(f"[提示] 连续标签按中位数 {cut:.3f} 二分为 低/高。")
        else:
            y_raw = pd.qcut(y_raw, 3, labels=["低", "中", "高"], duplicates="drop")
            print("[提示] 连续标签按三分位分箱为 低/中/高。")
    valid = y_raw.notna()
    le = LabelEncoder()
    le.fit(y_raw.loc[valid].astype(str))
    y = np.full(len(df), -1, dtype=int)
    y[valid.to_numpy()] = le.transform(y_raw.loc[valid].astype(str))
    print(f"标签 '{cfg.target}'：{len(le.classes_)} 类 {list(le.classes_)}，"
          f"有效 {int(valid.sum())}/{len(df)} 条")
    return y, le, y_raw


def compute_vif(X):
    """方差膨胀因子：VIF_j = 1/(1-R_j²)，R_j² 来自 x_j 对其余特征的线性回归。"""
    vals = X.to_numpy(dtype=float)
    rows = []
    for j, name in enumerate(X.columns):
        A = np.column_stack([np.ones(len(vals)), np.delete(vals, j, axis=1)])
        beta, *_ = np.linalg.lstsq(A, vals[:, j], rcond=None)
        ss_res = float(((vals[:, j] - A @ beta) ** 2).sum())
        ss_tot = float(((vals[:, j] - vals[:, j].mean()) ** 2).sum())
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        vif = 1 / (1 - r2) if r2 < 0.9999 else np.inf
        rows.append(dict(特征=name, R2=round(r2, 4),
                         VIF=round(vif, 3) if np.isfinite(vif) else np.inf,
                         判断="共线严重" if vif > 10 else ("需留意" if vif > 5 else "正常")))
    return pd.DataFrame(rows).sort_values("VIF", ascending=False).reset_index(drop=True)


def wald_coef_table(X, y, model_infer, le):
    """二分类逻辑回归的系数 / OR / 95%CI / Wald p 值。

    model_infer 用 penalty=None（近似极大似然估计），Wald 标准误才有意义；
    多分类时只给各类系数与 OR，不做 Wald 检验（需 OvR 或多项 logit 框架）。
    """
    names = ["截距"] + list(X.columns)
    if len(le.classes_) != 2:
        tab = pd.DataFrame(model_infer.coef_.T, index=list(X.columns),
                           columns=[f"coef_{c}" for c in le.classes_])
        tab["OR均值"] = np.exp(tab.mean(axis=1))
        print(f"[提示] {len(le.classes_)} 分类，跳过 Wald 检验（OR 需按 OvR 逐类解释）。")
        return tab.reset_index().rename(columns={"index": "特征"})
    coef = np.r_[model_infer.intercept_[0], model_infer.coef_.ravel()]
    p = model_infer.predict_proba(X)[:, 1]
    Xd = np.column_stack([np.ones(len(X)), X.to_numpy(float)])
    W = np.clip(p * (1 - p), 1e-8, None)        # 防止极端概率导致权重为 0
    cov = np.linalg.pinv((Xd * W[:, None]).T @ Xd)
    se = np.sqrt(np.clip(np.diag(cov), 0, None))
    z = np.divide(coef, se, out=np.zeros_like(coef), where=se > 0)
    pv = 2 * stats.norm.sf(np.abs(z))
    tab = pd.DataFrame(dict(
        特征=names, 系数beta=np.round(coef, 4), 标准误SE=np.round(se, 4),
        z值=np.round(z, 3), p值=np.round(pv, 5),
        OR=np.round(np.exp(coef), 4),
        OR下限=np.round(np.exp(coef - 1.96 * se), 4),
        OR上限=np.round(np.exp(coef + 1.96 * se), 4)))
    tab["显著性"] = np.where(tab.p值 < 0.01, "***",
                     np.where(tab.p值 < 0.05, "**",
                     np.where(tab.p值 < 0.1, "*", "")))
    tab["结论"] = np.where(tab.特征 == "截距", "—（截距，不解释 OR）",
                    np.where(tab.OR > 1, "促进事件发生", "抑制事件发生"))
    return tab


# 在当前数据上试算
y, LE, Y_RAW = prepare_target(df, CONFIG)
mask = y >= 0
Xm, ym = X.loc[mask], y[mask]
vif = compute_vif(Xm)
print("\n===== VIF 共线性诊断（前 10）=====")
print(vif.head(10).to_string(index=False))
model_infer = LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000,
                                 random_state=CONFIG.random_state).fit(Xm, ym)
coef_tab = wald_coef_table(Xm, ym, model_infer, LE)
print("\n===== 系数 / OR / Wald 检验 =====")
print(coef_tab.to_string(index=False))

[提示] 连续标签按中位数 69.736 二分为 低/高。
标签 'survey_score'：2 类 ['低', '高']，有效 348/366 条

===== VIF 共线性诊断（前 10）=====
                 特征     R2    VIF   判断
          region=西部 0.8302 5.8900 需留意
                 pm25 0.7351 3.7760   正常
          region=南部 0.7261 3.6500   正常
          region=北部 0.5773 2.3660   正常
         pm25_missing 0.3467 1.5310   正常
         fuel_type=煤 0.2187 1.2800   正常
       fuel_type=电力 0.1937 1.2400   正常
     fuel_type=生物质 0.1690 1.2030   正常
  temperature_missing 0.0744 1.0800   正常
traffic_index_missing 0.0449 1.0470   正常

===== 系数 / OR / Wald 检验 =====
                 特征  系数beta  标准误SE     z值    p值     OR  OR下限  OR上限 显著性                 结论
                 截距    1.1720    0.4214  2.7810 0.0054 3.2285  1.4135  7.3737    *** —（截距，不解释 OR）
                 pm25   -0.2474    0.2191 -1.1290 0.2589 0.7808  0.5082  1.1998                抑制事件发生
          temperature   -0.0420    0.1143 -0.3670 0.7133 0.9589  0.7665  1.1996                抑制事件发生
        traffic_index   -0.0349    0.

### 6. 交叉验证、留出集评估与阈值优化

先用 5 个正则强度 C 做分层交叉验证（灵敏度分析），再在留出集上给完整指标、
ROC 曲线、混淆矩阵；二分类额外给出 F1 最大 / Youden J 最大两个最优阈值。

In [14]:
# ============================================================
# 6. 交叉验证 + 留出集评估 + 阈值优化
# ============================================================
def make_model(cfg):
    """带正则的预测模型（class_weight 处理类别不平衡）。"""
    return LogisticRegression(C=cfg.lr_C, penalty="l2", solver="lbfgs",
                              class_weight=cfg.lr_class_weight,
                              max_iter=5000, random_state=cfg.random_state)


def evaluate_logistic(Xm, ym, le, cfg):
    n_splits = min(cfg.n_splits, int(pd.Series(ym).value_counts().min()))
    cv = StratifiedKFold(n_splits=max(2, n_splits), shuffle=True,
                         random_state=cfg.random_state)
    # 6.1 不同正则强度横向比较（论文里的灵敏度分析）
    rows = []
    for C in (0.01, 0.1, 1.0, 10.0, 100.0):
        m = LogisticRegression(C=C, penalty="l2", solver="lbfgs",
                               class_weight=cfg.lr_class_weight,
                               max_iter=5000, random_state=cfg.random_state)
        acc = cross_val_score(m, Xm, ym, cv=cv, scoring="accuracy")
        f1 = cross_val_score(m, Xm, ym, cv=cv, scoring="f1_macro")
        auc = cross_val_score(m, Xm, ym, cv=cv, scoring="roc_auc_ovr")
        rows.append(dict(C=C, CV_Accuracy=acc.mean(), CV_Accuracy_std=acc.std(),
                         CV_F1=f1.mean(), CV_AUC=auc.mean()))
    cv_tab = pd.DataFrame(rows)
    print("===== 正则强度 C 的交叉验证对比 =====")
    print(cv_tab.to_string(index=False))

    # 6.2 留出集
    model = make_model(cfg)
    X_tr, X_te, y_tr, y_te = train_test_split(
        Xm, ym, test_size=cfg.test_size, stratify=ym, random_state=cfg.random_state)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)
    metrics = dict(模型="逻辑回归", Accuracy=accuracy_score(y_te, pred),
                   Macro_Precision=precision_score(y_te, pred, average="macro", zero_division=0),
                   Macro_Recall=recall_score(y_te, pred, average="macro", zero_division=0),
                   Macro_F1=f1_score(y_te, pred, average="macro", zero_division=0),
                   ROC_AUC=(roc_auc_score(y_te, proba[:, 1]) if len(le.classes_) == 2
                            else roc_auc_score(y_te, proba, multi_class="ovr")))
    print("\n===== 留出集指标 =====")
    print(pd.DataFrame([metrics]).to_string(index=False))
    all_labels = list(range(len(le.classes_)))
    print("\n分类报告：")
    print(classification_report(y_te, pred, labels=all_labels,
                                target_names=le.classes_, zero_division=0))

    # 6.3 阈值优化（只针对二分类）
    thr_tab = None
    if len(le.classes_) == 2:
        p1 = proba[:, 1]
        fpr, tpr, roc_thr = roc_curve(y_te, p1)
        j = tpr - fpr
        thr_youden = float(roc_thr[int(np.argmax(j))])
        pr, rc, prc_thr = precision_recall_curve(y_te, p1)
        f1s = 2 * pr * rc / (pr + rc + 1e-12)
        thr_f1 = float(prc_thr[int(np.argmax(f1s[:-1]))]) if len(prc_thr) else 0.5
        thr_tab = pd.DataFrame([
            dict(准则="默认 0.5", 阈值=0.5,
                 F1=f1_score(y_te, (p1 >= 0.5).astype(int), zero_division=0)),
            dict(准则="F1 最大", 阈值=round(thr_f1, 4),
                 F1=f1_score(y_te, (p1 >= thr_f1).astype(int), zero_division=0)),
            dict(准则="Youden J 最大", 阈值=round(thr_youden, 4),
                 F1=f1_score(y_te, (p1 >= thr_youden).astype(int), zero_division=0))])
        print("\n===== 二分类阈值选择 =====")
        print(thr_tab.to_string(index=False))

    # 6.4 图：ROC + 混淆矩阵 + 系数森林图 + VIF
    if cfg.make_plots:
        fig, ax = plt.subplots(figsize=(6, 4.6))
        for i, cls in enumerate(le.classes_):
            fpr, tpr, _ = roc_curve((y_te == i).astype(int), proba[:, i])
            ax.plot(fpr, tpr, lw=2, label=f"{cls} (AUC={roc_auc_score((y_te == i).astype(int), proba[:, i]):.3f})")
        ax.plot([0, 1], [0, 1], "--", color="#888")
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC 曲线")
        ax.legend(fontsize=9); fig.tight_layout(); save_fig(fig, "ROC曲线", cfg)

        fig, ax = plt.subplots(figsize=(5.2, 4.2))
        ConfusionMatrixDisplay.from_predictions(y_te, pred, labels=all_labels,
                                                display_labels=le.classes_,
                                                cmap="Blues", ax=ax, colorbar=False)
        ax.set_title("混淆矩阵"); fig.tight_layout(); save_fig(fig, "混淆矩阵", cfg)

        ct = coef_tab[coef_tab.特征 != "截距"].head(14).iloc[::-1]
        if "OR" in ct.columns and "OR下限" in ct.columns:
            fig, ax = plt.subplots(figsize=(7, max(3, 0.38 * len(ct))))
            ax.errorbar(ct.OR, range(len(ct)),
                        xerr=[ct.OR - ct.OR下限, ct.OR上限 - ct.OR],
                        fmt="o", color="#185FA5", ecolor="#85B7EB", capsize=3)
            ax.axvline(1, ls="--", color="#d95f5f")
            ax.set_yticks(range(len(ct))); ax.set_yticklabels(ct.特征)
            ax.set_xlabel("优势比 OR（含 95% 置信区间）")
            ax.set_title("系数森林图：OR=1 表示无影响")
            fig.tight_layout(); save_fig(fig, "OR森林图", cfg)

        v = vif.head(14).iloc[::-1]
        fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(v))))
        ax.barh(v.特征, v.VIF.replace([np.inf], 999).clip(upper=50), color="#4a9d6a")
        ax.axvline(10, ls="--", color="#d95f5f", label="VIF=10 警戒线")
        ax.set_xlabel("VIF"); ax.set_title("共线性诊断"); ax.legend(fontsize=8)
        fig.tight_layout(); save_fig(fig, "VIF", cfg)

    return dict(cv_table=cv_tab, metrics=metrics, thr_table=thr_tab,
                model=model, classification_report=classification_report(
                    y_te, pred, labels=all_labels, target_names=le.classes_,
                    zero_division=0))


EVAL = evaluate_logistic(Xm, ym, LE, CONFIG)

===== 正则强度 C 的交叉验证对比 =====
       C  CV_Accuracy  CV_Accuracy_std  CV_F1  CV_AUC
  0.0100       0.5403           0.0424 0.5378  0.5278
  0.1000       0.5432           0.0548 0.5395  0.5610
  1.0000       0.5576           0.0608 0.5548  0.5860
 10.0000       0.5718           0.0664 0.5686  0.5945
100.0000       0.5718           0.0620 0.5698  0.5951

===== 留出集指标 =====
    模型  Accuracy  Macro_Precision  Macro_Recall  Macro_F1  ROC_AUC
逻辑回归    0.6092           0.6104        0.6097    0.6087   0.6268

分类报告：
              precision    recall  f1-score   support

           低       0.62      0.57      0.60        44
           高       0.60      0.65      0.62        43

    accuracy                           0.61        87
   macro avg       0.61      0.61      0.61        87
weighted avg       0.61      0.61      0.61        87


===== 二分类阈值选择 =====
         准则   阈值     F1
     默认 0.5 0.5000 0.6222
      F1 最大 0.3395 0.7027
Youden J 最大 0.4619 0.6923
[图片] 逻辑回归_满意度高低_ROC曲线.png（同名覆盖）
[图片] 逻辑回归

### 结果怎么拿

`bundle` 装着所有中间对象，常用的几个键见下面各节；同时结果会写成
`输出数据/` 下以 `逻辑回归` 开头、带 tag 的 `_结果明细.csv`（原表 + 预测/得分）
与 `_结果汇总.xlsx`（多 Sheet）；图片单独存 PNG。
换 tag 会新增一组文件，同 tag 是覆盖。

In [16]:
# ============================================================
# 7. 结果导出 + 新数据推理
# ============================================================
def export_results(bundle, cfg):
    """CSV 明细 + 多 Sheet Excel；图片在前述各节单独保存。"""
    out = bundle["df"].copy()
    out["y_true"] = bundle["y_raw"].astype(str).values
    # 标签缺失的样本没参与训练，预测结果要按有效样本的行索引对齐，不能直接整列赋值
    idx = bundle["valid_index"]
    out["pred_class"] = ""
    out.loc[idx, "pred_class"] = bundle["label_encoder"].inverse_transform(
        bundle["y_pred_all"])
    proba = bundle["model"].predict_proba(bundle["X"])
    for i, c in enumerate(bundle["label_encoder"].classes_):
        out[f"prob_{c}"] = np.nan
        out.loc[idx, f"prob_{c}"] = np.round(proba[:, i], 4)
    csv_path = OUT_DIR / f"{PREFIX}_{cfg.tag}_结果明细.csv"
    out.to_csv(csv_path, index=False, encoding="utf-8-sig")

    sheets = {"数据体检": bundle["audit"], "系数与OR": bundle["coef_tab"],
              "VIF": bundle["vif"], "正则强度对比": bundle["eval"]["cv_table"],
              "留出集指标": pd.DataFrame([bundle["eval"]["metrics"]])}
    if bundle["eval"]["thr_table"] is not None:
        sheets["阈值选择"] = bundle["eval"]["thr_table"]
    xlsx = OUT_DIR / f"{PREFIX}_{cfg.tag}_结果汇总.xlsx"
    try:
        with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
            for n, t in sheets.items():
                t.to_excel(w, sheet_name=n[:31], index=False)
        print(f"[Excel] {xlsx.name}（{len(sheets)} 个工作表）")
    except ImportError:
        print("[提示] 缺 openpyxl，跳过 Excel；CSV 已导出。")
    print(f"[CSV]   {csv_path.name}")
    return csv_path, xlsx


def predict_new(bundle, df_new):
    """新数据推理：复用同一个预处理器与模型，输出预测类别和各分类概率。"""
    Xn = bundle["preprocessor"].transform(df_new)
    proba = bundle["model"].predict_proba(Xn)
    out = df_new.copy()
    out["pred_class"] = bundle["label_encoder"].inverse_transform(bundle["model"].predict(Xn))
    for i, c in enumerate(bundle["label_encoder"].classes_):
        out[f"prob_{c}"] = np.round(proba[:, i], 4)
    print(f"新数据 {len(out)} 行完成推理")
    return out

### 一键入口 `run_pipeline` / `run_on`

`run_on("文件名", target=..., tag=...)` 是最省事的写法：自动定位文件、检查存在性、
校验参数名、创建配置、跑完整流程并返回 `bundle`。想逐项调参就用 `Config + run_pipeline`。

In [18]:
# ============================================================
# 8. 一键流程 + run_on
# ============================================================
def run_pipeline(cfg):
    print("=" * 68)
    print(f"逻辑回归：{cfg.tag} | 数据 {cfg.data_path} | 标签 {cfg.target}")
    print("=" * 68)
    d, meta = load_table(cfg)
    if cfg.target is None or cfg.target not in d.columns:
        raise KeyError(f"逻辑回归需要标签列；'{cfg.target}' 不存在。当前列名：{list(d.columns)}")
    aud = data_audit(d, cfg)
    p = TabularPreprocessor(cfg).fit(d)
    Xv = p.transform(d)
    yv, le, yraw = prepare_target(d, cfg)
    m = yv >= 0
    Xm, ym = Xv.loc[m], yv[m]
    print(f"特征矩阵 {Xm.shape}；有效样本 {len(ym)}；剔除列 {p.dropped_cols_ or '无'}")

    vif = compute_vif(Xm)
    infer = LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000,
                               random_state=cfg.random_state).fit(Xm, ym)
    coef = wald_coef_table(Xm, ym, infer, le)
    print("\n系数 / OR 表：")
    print(coef.to_string(index=False))
    ev = evaluate_logistic(Xm, ym, le, cfg)

    # 最终模型在全量有效样本上重拟合，供 predict_new 使用
    final = make_model(cfg).fit(Xm, ym)
    bundle = dict(cfg=cfg, meta=meta, df=d, audit=aud, preprocessor=p, X=Xm,
                  valid_index=Xm.index, y=ym, y_raw=yraw, label_encoder=le,
                  coef_tab=coef, vif=vif, infer_model=infer, eval=ev,
                  model=final, y_pred_all=final.predict(Xm))
    export_results(bundle, cfg)
    print("=" * 68)
    print(f"完成：{cfg.tag} | 留出集 Macro-F1 = "
          f"{ev['metrics']['Macro_F1']:.4f} | 结果见 {OUT_DIR}")
    print("=" * 68)
    return bundle


def run_on(data_path, target, tag="lr_demo", **overrides):
    """一行换数据：bundle = run_on("新数据.csv", target="标签列", tag="本次名字")"""
    path = Path(data_path)
    if not path.is_absolute():
        path = (ROOT / path) if (path.parts and path.parts[0] == DATA_DIR.name) else (DATA_DIR / path)
    if not path.exists():
        raise FileNotFoundError(f"找不到数据文件：{path}\n请把文件放进 {DATA_DIR}。")
    valid = {f.name for f in fields(Config)}
    bad = set(overrides) - valid
    if bad:
        raise ValueError(f"未知配置项 {sorted(bad)}；可用字段：{sorted(valid)}")
    params = dict(data_path=path, target=target, tag=tag)
    params.update(overrides)
    # 从当前 CONFIG 继承未显式覆盖的字段（例如逻辑回归默认丢弃独热参照组），
    # 这样「改 CONFIG 就生效」与「用 run_on 一行换数据」两条路径行为一致
    return run_pipeline(replace(CONFIG, **params))

### 主执行（默认运行这一格，就出一套完整结果）

把下面这一行的三个值换成你的文件名、标签列、tag，其余都不用动。

In [20]:
# ============================================================
# 主执行：换题目时只改这一行
# ============================================================

bundle = run_on("mcm_final_raw.csv", target="survey_score", tag="满意度高低")
print("bundle 可用键：", sorted(bundle.keys()))

逻辑回归：满意度高低 | 数据 D:\git项目\导入数据\mcm_final_raw.csv | 标签 survey_score
读取成功 : mcm_final_raw.csv（utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删重复 15 行
文本转数值 : ['pm25']；日期列：['date']；哨兵转缺失：{'traffic_index': 6}
===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    s

### 可选示例（默认不执行，把 RUN_EXAMPLES 改成 True 才会跑）

演示换数据、换参数、新数据推理；跑起来会多产出一小组文件。

In [22]:
if RUN_EXAMPLES:
    # 示例 1：换一份 Excel、换成地区标签（多分类），不生成图片
    b2 = run_on(OUT_DIR / "final_cleaned.csv", target="region", tag="地区多分类",
                make_plots=False, lr_binary_split=False)
    # 示例 2：用训练好的模型预测新样本
    new_pred = predict_new(bundle, bundle["df"].head(8))
    print(new_pred[["pred_class"]].to_string())
    # 示例 3：清理某一组历史结果
    # clean_outputs("地区多分类")
else:
    print("已跳过可选示例（RUN_EXAMPLES=False）。")

已跳过可选示例（RUN_EXAMPLES=False）。
